# PyTorch ile Evrişimsel Sinir Ağı (CNN) Uygulaması

Bu notebook, PyTorch kütüphanesini kullanarak sıfırdan bir Evrişimsel Sinir Ağı (CNN) kurmayı, eğitmeyi ve test etmeyi amaçlamaktadır. Uygulamada, popüler "Cats vs. Dogs" veri seti kullanılacaktır.


### 1. Gerekli Kütüphanelerin Yüklenmesi

Projemizde kullanacağımız kütüphaneleri yüklüyoruz:
- **torch ve torch.nn**: Sinir ağı katmanları, modelleri ve temel tensör işlemleri için.
- **torch.optim**: Modeli eğitmek için optimizasyon algoritmaları (örn. Adam).
- **torch.utils.data**: Veri setlerini yönetmek ve yüklemek için `Dataset` ve `DataLoader`.
- **torchvision.transforms**: Görüntülere uygulanacak ön işleme ve veri artırma adımları için.
- **glob**: Dosya yollarını bulmak için.
- **numpy ve PIL**: Görüntü işleme ve sayısal operasyonlar için.


In [5]:
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import transforms

from glob import glob
import numpy as np
from PIL import Image

### 2. PyTorch `nn.Conv2d` Katmanını Anlamak

CNN mimarilerinin temel taşı olan evrişim katmanının PyTorch'ta nasıl tanımlandığını ve hangi parametrelere sahip olduğunu inceleyelim.

**Örnek 1:** 1 giriş kanallı (siyah-beyaz), 3 çıkış kanallı (3 farklı filtre uygulanmış) ve 5x5 boyutunda bir filtreye sahip evrişim katmanı. `named_parameters()` ile bu katmanın öğrenilebilir parametrelerini (ağırlıklar ve bias) ve boyutlarını görebiliriz.


In [6]:
layer = nn.Conv2d(1, 3, kernel_size=5, stride=1, padding=1)

for name, param in layer.named_parameters():
    if param.requires_grad:
        print(name, param.shape)

weight torch.Size([3, 1, 5, 5])
bias torch.Size([3])


In [7]:
layer = nn.Conv2d(3, 8, kernel_size=5, stride=1, padding=1)

for name, param in layer.named_parameters():
    if param.requires_grad:
        print(name, param.shape)

weight torch.Size([8, 3, 5, 5])
bias torch.Size([8])


In [8]:
layer = nn.Sequential(
    nn.Conv2d(3, 8, kernel_size=5, stride=1, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2))

for name, param in layer.named_parameters():
    if param.requires_grad:
        print(name, param.shape)

0.weight torch.Size([8, 3, 5, 5])
0.bias torch.Size([8])


In [9]:
layer = nn.Conv2d(3, 8, kernel_size=5, stride=1, padding=1)
dummy_img = torch.randn(3, 50, 40)

output = layer(dummy_img)

print(output.size())

torch.Size([8, 48, 38])


In [10]:
layer = nn.Sequential(
    nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2))
dummy_img = torch.randn(3, 50, 40)

output = layer(dummy_img)

print(output.size())

torch.Size([32, 25, 20])


### 4. Veri Setinin Hazırlanması

Veri seti: https://www.kaggle.com/datasets/samuelcortinhas/cats-and-dogs-image-classification

#### a. Dosya Yollarının Toplanması

`glob` kütüphanesi ile eğitim ve test setlerindeki kedi ve köpek görüntülerinin dosya yollarını topluyoruz ve bir sözlük yapısında saklıyoruz.


veri seti: https://www.kaggle.com/datasets/samuelcortinhas/cats-and-dogs-image-classification


In [11]:
train_dir_cats = "kedi_kopek/train/cats/*"
train_dir_dogs = "kedi_kopek/train/dogs/*"

train_dict = {"cats": [], "dogs": []}
for cat_file in glob(train_dir_cats):
    train_dict["cats"].append(cat_file)

for dog_file in glob(train_dir_dogs):
    train_dict["dogs"].append(dog_file)

test_dir_cats = "kedi_kopek/test/cats/*"
test_dir_dogs = "kedi_kopek/test/dogs/*"

test_dict = {"cats": [], "dogs": []}
for cat_file in glob(test_dir_cats):
    test_dict["cats"].append(cat_file)

for dog_file in glob(test_dir_dogs):
    test_dict["dogs"].append(dog_file)

#### c. Özel Veri Seti Sınıfı (Custom Dataset)

PyTorch'un `Dataset` sınıfını kalıtım alarak kendi veri seti sınıfımızı oluşturuyoruz. Bu sınıf, bir indekse karşılık gelen görüntüyü ve etiketi döndürmekten sorumludur. `__getitem__` metodu içinde ayrıca görüntülere uygulanacak dönüşümleri (`transforms`) de tanımlıyoruz.

**Dönüşümler (Transforms)**:
- `Resize`: Görüntüleri standart bir boyuta getirir.
- `RandomCrop`: Görüntüden rastgele bir bölgeyi keserek modelin dayanıklılığını artırır.
- `RandomHorizontalFlip`: Görüntüyü rastgele yatayda çevirerek veri artırma (data augmentation) yapar.
- `ToTensor`: Görüntüyü PyTorch tensörüne dönüştürür.


In [12]:
file_paths = []
labels = np.zeros(len(train_dict["cats"]) + len(train_dict["dogs"]))
labels[len(train_dict["cats"]):] = 1
file_paths.extend(train_dict["cats"])
file_paths.extend(train_dict["dogs"])
for i in [0, 200, 400, 500]:
    print(file_paths[i], labels[i])

kedi_kopek/train/cats/cat_10.jpg 0.0
kedi_kopek/train/cats/cat_489.jpg 0.0
kedi_kopek/train/dogs/dog_332.jpg 1.0
kedi_kopek/train/dogs/dog_529.jpg 1.0


In [13]:
class CatsDogsDataset(Dataset):
    def __init__(self, files_dict):
        self.file_paths = []
        self.labels = np.zeros(
            len(train_dict["cats"]) + len(train_dict["dogs"]))
        self.labels[len(files_dict["cats"]):] = 1
        self.file_paths.extend(files_dict["cats"])
        self.file_paths.extend(files_dict["dogs"])
        self.transforms = transforms.Compose([
            transforms.Resize(256),
            transforms.RandomCrop(224),
            transforms.RandomHorizontalFlip(),
            transforms.Resize(128),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, index):
        img = Image.open(self.file_paths[index])
        img = img.convert("RGB")
        img = self.transforms(img)
        label = labels[index]
        return img.numpy().astype("float32"), label.astype("long")

In [14]:
train_dataset = CatsDogsDataset(train_dict)
BATCH_SIZE = 16
train_data_loader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True)

In [15]:
batch = next(iter(train_data_loader))[0]
print("Batch Shape:  ", batch.shape)

layer1 = nn.Sequential(
    nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2))
x = layer1(batch)
print("After Layer 1:", x.shape)

layer2 = nn.Sequential(
    nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2))
x = layer2(x)
print("After Layer 2:", x.shape)

Batch Shape:   torch.Size([16, 3, 128, 128])
After Layer 1: torch.Size([16, 32, 64, 64])
After Layer 2: torch.Size([16, 64, 32, 32])


In [16]:
# NVIDIA GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [17]:
class CNN_1(nn.Module):

    def __init__(self):
        super().__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2))
        self.layer2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2))
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 32 * 32, 2))

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.fc(x)
        return x

In [18]:
model_1 = CNN_1().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = Adam(model_1.parameters(), lr=0.001)

In [19]:
num_batches = len(train_data_loader)

for epoch in range(10):
    avg_loss = 0

    for X, Y in train_data_loader:
        X = X.to(device)
        Y = Y.to(device)

        y_hat = model_1(X)
        loss = criterion(y_hat, Y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        avg_loss += loss.item()

    avg_loss = avg_loss / num_batches

    print(f"Epoch: {epoch+1}, Loss: {avg_loss:.4f}")

Epoch: 1, Loss: 0.8384
Epoch: 2, Loss: 0.6927
Epoch: 3, Loss: 0.6930
Epoch: 4, Loss: 0.6932
Epoch: 5, Loss: 0.6913
Epoch: 6, Loss: 0.6899
Epoch: 7, Loss: 0.6891
Epoch: 8, Loss: 0.6928
Epoch: 9, Loss: 0.6861
Epoch: 10, Loss: 0.6835


In [20]:
test_dataset = CatsDogsDataset(test_dict)
test_data_loader = DataLoader(
    dataset=test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=True)

with torch.no_grad():
    model_1.to("cpu")
    total_predictions = 0
    num_correct_predictions = 0
    for X, Y in test_data_loader:
        predictions = model_1(X)
        predictions = predictions.argmax(dim=1)
        num_correct_predictions += (predictions == Y).sum()
        total_predictions += len(Y)

    print(num_correct_predictions.item(), "/", total_predictions)   # 56 / 128

0 / 128


In [21]:
batch = next(iter(train_data_loader))[0]
print(batch.shape)

conv_layer_1 = nn.Sequential(
    nn.Conv2d(3, 64, 3, padding=1),
    nn.ReLU(),
    nn.BatchNorm2d(64),
    nn.MaxPool2d(2))
x = conv_layer_1(batch)
print("L1:", x.shape)

conv_layer_2 = nn.Sequential(
    nn.Conv2d(64, 512, 3, padding=1),
    nn.ReLU(),
    nn.BatchNorm2d(512),
    nn.MaxPool2d(2))
x = conv_layer_2(x)
print("L2:", x.shape)

conv_layer_3 = nn.Sequential(
    nn.Conv2d(512, 512, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.BatchNorm2d(512),
    nn.MaxPool2d(2))
x = conv_layer_3(x)
print("L3 - 1:", x.shape)

x = conv_layer_3(x)
print("L3 - 2:", x.shape)

x = conv_layer_3(x)
print("L3 - 3:", x.shape)

x = conv_layer_3(x)
print("L3 - 4:", x.shape)

torch.Size([16, 3, 128, 128])
L1: torch.Size([16, 64, 64, 64])
L2: torch.Size([16, 512, 32, 32])
L3 - 1: torch.Size([16, 512, 16, 16])
L3 - 2: torch.Size([16, 512, 8, 8])
L3 - 3: torch.Size([16, 512, 4, 4])
L3 - 4: torch.Size([16, 512, 2, 2])


### 6. Daha Derin bir CNN Mimarisi Oluşturma (`CNN_2`)

Şimdi, VGG gibi daha derin ağlardan esinlenerek daha karmaşık bir model (`CNN_2`) oluşturacağız. Bu model, ardışık evrişim katmanları ve `BatchNorm2d` gibi normalizasyon katmanları içerecektir.

#### a. Model Mimarisi Boyut Kontrolü

Yeni ve daha derin mimarimizin katmanlarından bir veri grubunu geçirerek boyut değişimlerini tekrar kontrol ediyoruz. Özellikle tekrarlayan blokların (`conv_layer_3`) boyutları nasıl etkilediğine dikkat edelim.


In [22]:
class CNN_2(torch.nn.Module):

    def __init__(self):
        super().__init__()
        self.conv_layer_1 = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(2))
        self.conv_layer_2 = nn.Sequential(
            nn.Conv2d(64, 512, 3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(512),
            nn.MaxPool2d(2))
        self.conv_layer_3 = nn.Sequential(
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(512),
            nn.MaxPool2d(2))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=512*2*2, out_features=2))

    def forward(self, x):
        x = self.conv_layer_1(x)
        x = self.conv_layer_2(x)
        x = self.conv_layer_3(x)
        x = self.conv_layer_3(x)
        x = self.conv_layer_3(x)
        x = self.conv_layer_3(x)
        x = self.classifier(x)
        return x

In [23]:
model_2 = CNN_2().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = Adam(model_2.parameters(), lr=0.01)

num_batches = len(train_data_loader)

for epoch in range(20):
    avg_loss = 0

    for X, Y in train_data_loader:
        X = X.to(device)
        Y = Y.to(device)

        y_hat = model_2(X)
        loss = criterion(y_hat, Y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        avg_loss += loss.item()

    avg_loss = avg_loss / num_batches

    print(f"Epoch: {epoch+1}, Loss: {avg_loss:.4f}")

Epoch: 1, Loss: 4.0935
Epoch: 2, Loss: 2.6852
Epoch: 3, Loss: 2.3407
Epoch: 4, Loss: 2.2955
Epoch: 5, Loss: 1.5088
Epoch: 6, Loss: 1.8902
Epoch: 7, Loss: 1.4585
Epoch: 8, Loss: 1.2936
Epoch: 9, Loss: 1.0279
Epoch: 10, Loss: 0.9575
Epoch: 11, Loss: 0.8645
Epoch: 12, Loss: 0.8852


KeyboardInterrupt: 

: 

In [ ]:
with torch.no_grad():
    model_2.to("cpu")
    model_2.eval()
    total_predictions = 0
    num_correct_predictions = 0
    for X, Y in test_data_loader:
        predictions = model_2(X)
        predictions = predictions.argmax(dim=1)
        num_correct_predictions += (predictions == Y).sum()
        total_predictions += len(Y)

    print(num_correct_predictions.item(), "/", total_predictions)   # 108 / 128